# WoundScope — staged Colab full run

這份 notebook 是 thin wrapper：掛載 private Drive、驗證 immutable source ZIP、強制 CUDA，然後只呼叫一次可恢復的 staged pipeline。Quick、comparison、loss selection、multi-seed final、locked official validation、ONNX/parity 與 safe handoff 全由可測試的 Python module 管理。

> Quick metrics 只作 smoke evidence。輸出是研究用 wound segmentation，不是診斷、嚴重度、預後或治療建議。FUSeg images、masks、manifest、gallery、weights 與 ONNX 只留在 private artifacts。

In [ ]:
#@title 1. Mount private Drive and locate the immutable source bundle
from google.colab import drive
from pathlib import Path
import os
runtime_root = Path(os.environ.setdefault('WOUNDSCOPE_RUNTIME_ROOT', str(Path.cwd()))).resolve()
os.chdir(runtime_root)
drive_mount = Path(os.environ.get('WOUNDSCOPE_DRIVE_MOUNT', str(runtime_root / 'drive')))
drive.mount(str(drive_mount))
source_zip = drive_mount / 'MyDrive' / 'WoundScope_colab_source.zip'
artifact_dir = drive_mount / 'MyDrive' / 'WoundScopeArtifacts'
if not source_zip.is_file():
    raise FileNotFoundError(f'Missing safe source ZIP: {source_zip}')
artifact_dir.mkdir(parents=True, exist_ok=True)
print('Source ZIP:', source_zip)
print('Private artifact root:', artifact_dir)

In [ ]:
#@title 2. Verify bundle_manifest.json and extract safely
import hashlib, json, shutil, zipfile
from pathlib import PurePosixPath
with zipfile.ZipFile(source_zip) as archive:
    names = [item.filename for item in archive.infolist() if not item.is_dir()]
    for name in names:
        path = PurePosixPath(name)
        if not name or '\\' in name or path.is_absolute() or '..' in path.parts:
            raise RuntimeError(f'Unsafe source archive path: {name!r}')
    if len(names) != len(set(names)) or 'bundle_manifest.json' not in names:
        raise RuntimeError('Invalid or duplicate source bundle inventory')
    manifest = json.loads(archive.read('bundle_manifest.json'))
    if manifest.get('kind') != 'source' or manifest.get('schema_version') != 1:
        raise RuntimeError('Incompatible source bundle schema')
    expected_names = {'bundle_manifest.json'}
    for record in manifest['files']:
        content = archive.read(record['path'])
        expected_names.add(record['path'])
        if len(content) != record['size'] or hashlib.sha256(content).hexdigest() != record['sha256']:
            raise RuntimeError(f'Source bundle checksum mismatch: {record["path"]}')
    if set(names) != expected_names:
        raise RuntimeError('Source ZIP contains unlisted members')
    source_commit = manifest['source_commit']
    project_dir = runtime_root / f'WoundScope_{source_commit[:12]}'
    if project_dir.exists():
        shutil.rmtree(project_dir)
    project_dir.mkdir(parents=True)
    for name in sorted(expected_names):
        target = project_dir.joinpath(*PurePosixPath(name).parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))
os.environ['WOUNDSCOPE_SOURCE_COMMIT'] = source_commit
print('Verified source commit:', source_commit)
print('Ephemeral project:', project_dir)

In [ ]:
#@title 3. Install the committed source and enforce the CUDA gate
import platform, subprocess, sys
os.chdir(project_dir)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train,export,app]'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required. Select a T4, L4, or A100 GPU runtime; CPU fallback is forbidden.')
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

In [ ]:
#@title 4. Run or resume every locked stage with one command
data_dir = Path(os.environ.get('WOUNDSCOPE_DATA_DIR', str(runtime_root / 'woundscope_data')))
os.environ['WOUNDSCOPE_DATA_DIR'] = str(data_dir)
os.environ['WOUNDSCOPE_ARTIFACT_DIR'] = str(artifact_dir)
command = [
    sys.executable, 'scripts/run_colab_pipeline.py',
    '--project-root', str(project_dir),
    '--data-dir', str(data_dir),
    '--artifact-dir', str(artifact_dir),
    '--source-commit', source_commit,
]
subprocess.run(command, check=True)
handoff = artifact_dir / 'handoff' / f'woundscope_colab_results_{source_commit[:12]}.zip'
if not handoff.is_file():
    raise RuntimeError('Pipeline completed without the required safe handoff ZIP')
print('SAFE_HANDOFF_READY:', handoff)